# Aspect Category Classification with Classical Machine Learning

This notebook compares classical machine-learning approaches for sentence-level aspect-category classification. The task is multi-label classification over six restaurant aspect categories.

Four text representations are evaluated:

- CountVectorizer;
- TF-IDF;
- Word2Vec sentence embeddings;
- FastText sentence embeddings.

Five classifiers are compared:

- Support Vector Machine;
- Random Forest;
- Multinomial Naive Bayes;
- Gradient Boosting;
- Logistic Regression.

Model quality is measured with weighted precision, recall, and F1.


## 1. Setup


In [ ]:
!pip install gensim
!python -m spacy download ru_core_news_md

import xml.etree.ElementTree as ET

import gensim
import numpy as np
import pandas as pd
import spacy

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.svm import SVC


In [ ]:
nlp = spacy.load("ru_core_news_md")

TRAIN_XML_PATH = "/content/drive/MyDrive/se16_ru_rest_train.xml"
WORD2VEC_MODEL_PATH = "/content/drive/MyDrive/model.bin"
FASTTEXT_MODEL_PATH = "/content/drive/MyDrive/fasttext 213/model.model"


## 2. Load sentence texts and aspect categories


In [ ]:
def load_category_dataset(xml_path):
    """Load sentence texts and top-level aspect categories from SemEval XML."""
    root = ET.parse(xml_path).getroot()

    texts = []
    categories = []

    for sentence in root.iter("sentence"):
        text_node = sentence.find("text")
        opinions = sentence.find("Opinions")

        if text_node is None:
            continue

        sentence_categories = []

        if opinions is not None:
            for opinion in opinions.findall("Opinion"):
                category = opinion.get("category")
                if category:
                    sentence_categories.append(category.split("#")[0])

        texts.append(text_node.text or "")
        categories.append(sorted(set(sentence_categories)))

    return texts, categories


texts, category_lists = load_category_dataset(TRAIN_XML_PATH)

CATEGORY_LABELS = [
    "FOOD",
    "SERVICE",
    "RESTAURANT",
    "AMBIENCE",
    "DRINKS",
    "LOCATION",
]

label_binarizer = MultiLabelBinarizer(classes=CATEGORY_LABELS)
label_binarizer.fit([CATEGORY_LABELS])

encoded_categories = label_binarizer.transform(category_lists)


## 3. Text preprocessing and train/validation split

Texts are lemmatized with spaCy and punctuation is removed. The annotated training collection is split into training and validation subsets in a 4:1 ratio (`test_size=0.2`, `random_state=42`).


In [ ]:
def preprocess_text(text):
    """Lemmatize text with spaCy and remove punctuation."""
    doc = nlp(text)

    return " ".join(
        token.lemma_
        for token in doc
        if token.lemma_ != "null"
        and token.pos_ != "PUNCT"
        and token.lemma_ != " "
    )


processed_texts = [
    preprocess_text(text)
    for text in texts
]

(
    train_texts,
    validation_texts,
    y_train,
    y_validation,
) = train_test_split(
    processed_texts,
    encoded_categories,
    test_size=0.2,
    random_state=42,
)


## 4. Text representations


In [ ]:
count_vectorizer = CountVectorizer()
tfidf_vectorizer = TfidfVectorizer()

X_train_count = count_vectorizer.fit_transform(train_texts)
X_validation_count = count_vectorizer.transform(validation_texts)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_validation_tfidf = tfidf_vectorizer.transform(validation_texts)

word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
    WORD2VEC_MODEL_PATH,
    binary=True,
)
fasttext_model = gensim.models.KeyedVectors.load(
    FASTTEXT_MODEL_PATH
)


def sentence_embedding(text, embedding_model):
    """Represent a sentence by the mean of its available word vectors."""
    vectors = [
        embedding_model[word]
        for word in text.split()
        if word in embedding_model
    ]

    if vectors:
        return np.mean(vectors, axis=0)

    return np.zeros(embedding_model.vector_size)


def build_embedding_matrix(texts, embedding_model):
    """Create one mean embedding vector per sentence."""
    return np.vstack(
        [
            sentence_embedding(text, embedding_model)
            for text in texts
        ]
    )


X_train_word2vec = build_embedding_matrix(
    train_texts,
    word2vec_model,
)
X_validation_word2vec = build_embedding_matrix(
    validation_texts,
    word2vec_model,
)

X_train_fasttext = build_embedding_matrix(
    train_texts,
    fasttext_model,
)
X_validation_fasttext = build_embedding_matrix(
    validation_texts,
    fasttext_model,
)


## 5. Evaluation

A `null` indicator is added for examples where none of the six aspect categories is active. Evaluation uses weighted precision, recall, and F1.


In [ ]:
def add_null_category(encoded_labels):
    """Append a null-category indicator for rows with no active aspect label."""
    null_column = (encoded_labels.sum(axis=1) == 0).astype(int).reshape(-1, 1)
    return np.hstack([encoded_labels, null_column])


def evaluate_multilabel_predictions(y_true, y_pred):
    """Compute weighted precision, recall, and F1."""
    y_true_with_null = add_null_category(np.asarray(y_true))
    y_pred_with_null = add_null_category(np.asarray(y_pred))

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_with_null,
        y_pred_with_null,
        average="weighted",
        zero_division=0,
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


## 6. Hyperparameter tuning

`GridSearchCV` is used to tune Multinomial Naive Bayes with 3-fold cross-validation over smoothing, class-prior, and prior-fitting settings.


In [ ]:
naive_bayes_param_grid = {
    "estimator__alpha": [
        0.1,
        0.5,
        1.0,
        1.5,
    ],
    "estimator__class_prior": [
        None,
        [0.2, 0.8],
    ],
    "estimator__fit_prior": [
        True,
        False,
    ],
}


def tune_multinomial_nb(
    X_train,
    y_train,
):
    """Tune Multinomial Naive Bayes with GridSearchCV."""
    classifier = MultiOutputClassifier(
        MultinomialNB()
    )

    grid_search = GridSearchCV(
        estimator=classifier,
        param_grid=naive_bayes_param_grid,
        cv=3,
    )
    grid_search.fit(
        X_train,
        y_train,
    )

    return grid_search


word2vec_scaler = MinMaxScaler()
X_train_word2vec_scaled = word2vec_scaler.fit_transform(
    X_train_word2vec
)
X_validation_word2vec_scaled = word2vec_scaler.transform(
    X_validation_word2vec
)

naive_bayes_grid_search = tune_multinomial_nb(
    X_train_word2vec_scaled,
    y_train,
)

naive_bayes_grid_search.best_params_


## 7. Classifier configurations

The experiments below evaluate representation-specific classifier configurations selected during model development.


In [ ]:
CLASSIFIER_CONFIGS = {
    "Count": {
        "SVM": SVC(
            C=1.0,
            kernel="linear",
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            min_samples_split=5,
            n_estimators=50,
            random_state=42,
        ),
        "Naive Bayes": MultinomialNB(
            alpha=1.0,
            class_prior=None,
            fit_prior=True,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=5,
            n_estimators=200,
            random_state=42,
        ),
        "Logistic Regression": LogisticRegression(
            C=1.0,
            max_iter=100,
        ),
    },
    "TF-IDF": {
        "SVM": SVC(
            C=1.0,
            kernel="linear",
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            min_samples_split=5,
            n_estimators=25,
            random_state=42,
        ),
        "Naive Bayes": MultinomialNB(
            alpha=1.0,
            class_prior=None,
            fit_prior=False,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=5,
            n_estimators=200,
            random_state=42,
        ),
        "Logistic Regression": LogisticRegression(
            C=1.0,
            max_iter=100,
        ),
    },
    "Word2Vec": {
        "SVM": SVC(
            C=10,
            kernel="rbf",
            gamma=0.01,
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            min_samples_split=5,
            n_estimators=25,
            random_state=42,
        ),
        "Naive Bayes": MultinomialNB(
            alpha=1.0,
            class_prior=None,
            fit_prior=False,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=5,
            n_estimators=200,
            random_state=42,
        ),
        "Logistic Regression": LogisticRegression(
            C=1.0,
            max_iter=100,
        ),
    },
    "FastText": {
        "SVM": SVC(
            C=10,
            kernel="rbf",
            gamma=0.1,
            random_state=42,
        ),
        "Random Forest": RandomForestClassifier(
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            min_samples_split=5,
            n_estimators=25,
            random_state=42,
        ),
        "Naive Bayes": MultinomialNB(
            alpha=1.0,
            class_prior=None,
            fit_prior=False,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=5,
            n_estimators=200,
            random_state=42,
        ),
        "Logistic Regression": LogisticRegression(
            C=1.0,
            max_iter=100,
        ),
    },
}


## 8. Run the experiments


In [ ]:
REPRESENTATIONS = {
    "Count": (
        X_train_count,
        X_validation_count,
    ),
    "TF-IDF": (
        X_train_tfidf,
        X_validation_tfidf,
    ),
    "Word2Vec": (
        X_train_word2vec,
        X_validation_word2vec,
    ),
    "FastText": (
        X_train_fasttext,
        X_validation_fasttext,
    ),
}


def prepare_features_for_classifier(
    representation_name,
    classifier_name,
    X_train,
    X_validation,
):
    """Apply representation-specific preprocessing for the classifier."""
    if classifier_name == "Naive Bayes" and representation_name in {
        "Word2Vec",
        "FastText",
    }:
        scaler = MinMaxScaler()
        return (
            scaler.fit_transform(X_train),
            scaler.transform(X_validation),
        )

    return X_train, X_validation


def run_classical_experiments():
    """Train classifier configurations and collect validation metrics."""
    results = []

    for representation_name, (
        X_train,
        X_validation,
    ) in REPRESENTATIONS.items():
        for classifier_name, base_classifier in CLASSIFIER_CONFIGS[
            representation_name
        ].items():
            train_features, validation_features = (
                prepare_features_for_classifier(
                    representation_name,
                    classifier_name,
                    X_train,
                    X_validation,
                )
            )

            classifier = MultiOutputClassifier(
                base_classifier
            )
            classifier.fit(
                train_features,
                y_train,
            )

            predictions = classifier.predict(
                validation_features
            )

            metrics = evaluate_multilabel_predictions(
                y_validation,
                predictions,
            )

            results.append(
                {
                    "representation": representation_name,
                    "classifier": classifier_name,
                    **metrics,
                }
            )

    return pd.DataFrame(results)


validation_results = run_classical_experiments()

validation_results.sort_values(
    "f1",
    ascending=False,
)
